# The Kernel Trick

**Companion lesson:** https://ml-viz.vercel.app/courses/svm/02-kernel-trick

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = '#94a3b8'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2e3347'

## Kernel Functions

In [ ]:
x1 = np.array([0, 0])
x2 = np.array([1, 1])

def linear_kernel(x1, x2): return x1 @ x2
def poly_kernel(x1, x2, c=1, d=2): return (x1 @ x2 + c)**d
def rbf_kernel(x1, x2, gamma=1): return np.exp(-gamma * np.linalg.norm(x1 - x2)**2)

print(f'Linear: {linear_kernel(x1, x2)}')
print(f'Poly (d=2): {poly_kernel(x1, x2)}')
print(f'RBF (γ=1): {rbf_kernel(x1, x2):.4f}')
print(f'RBF (γ=0.1): {rbf_kernel(x1, x2, gamma=0.1):.4f}')

## Linear vs RBF: Non-linearly separable data

In [ ]:
from sklearn.datasets import make_moons
from sklearn.svm import SVC

X, y = make_moons(n_samples=200, noise=0.15, random_state=42)
xx, yy = np.meshgrid(np.linspace(-2, 3, 200), np.linspace(-1.5, 2, 200))

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, kernel in zip(axes, ['linear', 'poly', 'rbf']):
    svm = SVC(kernel=kernel, C=1.0, gamma='scale')
    svm.fit(X, y)
    Z = svm.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdYlBu')
    ax.scatter(X[y == 0, 0], X[y == 0, 1], c='#f43f5e', s=15, alpha=0.7)
    ax.scatter(X[y == 1, 0], X[y == 1, 1], c='#818cf8', s=15, alpha=0.7)
    ax.set_title(f'{kernel.capitalize()} Kernel', color='white')
plt.suptitle('SVM with Different Kernels', color='white', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## The kernel trick: implicit feature maps

A kernel $K(x, x')$ computes a dot product in a high-dimensional space **without ever building it**. The RBF kernel corresponds to an infinite-dimensional space.

In [ ]:
# Explicit degree-2 map vs the polynomial kernel give the same dot product
def phi(x):  # map (a, b) -> (a^2, b^2, sqrt(2) a b, sqrt(2) a, sqrt(2) b, 1)
    a, b = x
    return np.array([a**2, b**2, np.sqrt(2)*a*b, np.sqrt(2)*a, np.sqrt(2)*b, 1])

x1, x2 = np.array([1.0, 2.0]), np.array([3.0, -1.0])
print('explicit  phi(x1).phi(x2):', phi(x1) @ phi(x2))
print('kernel    (x1.x2 + 1)^2 :', (x1 @ x2 + 1)**2)

## gamma controls RBF reach

Small `gamma` = smooth, far-reaching influence; large `gamma` = tight, wiggly boundaries that can overfit.

In [ ]:
from sklearn.svm import SVC
from sklearn.datasets import make_moons

X, y = make_moons(n_samples=200, noise=0.2, random_state=1)
xx, yy = np.meshgrid(np.linspace(-2, 3, 200), np.linspace(-1.5, 2, 200))
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, g in zip(axes, [0.1, 1, 30]):
    svm = SVC(kernel='rbf', gamma=g, C=1).fit(X, y)
    Z = svm.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdYlBu')
    ax.scatter(*X.T, c=y, cmap='RdYlBu', edgecolor='k', s=15)
    ax.set_title(f'gamma = {g}')
plt.tight_layout(); plt.show()

## Key takeaways

- Kernels let a linear SVM learn **non-linear** boundaries via implicit feature maps.
- **RBF** is the go-to general-purpose kernel; **polynomial** and **linear** are alternatives.
- `C` controls margin softness; `gamma` controls RBF reach — tune both together (grid search).
- Large `gamma` or `C` overfits; always scale features before kernel SVMs.